In [0]:
from pyspark.sql import functions as F
import json

In [0]:
def create_user_product_candidate_set(spark, run_date, db, table_names, user_lookback_win=30, product_lookback_win=180, num_shards=1):
    sdf = spark.sql("""
                    with active_users as (
                    select distinct user_id 
                    from {db}.{mmc_user_attributes}
                    where store_visits > 0 
                    and partition_date between date_sub('{run_date}', {user_lookback_win}) and '{run_date}'
                    ),
                    active_products as (
                    select distinct product_id
                    from {db}.{mmc_product_attributes}
                    where card_uv_imps > 0
                    and partition_date between date_sub('{run_date}', {product_lookback_win}) and '{run_date}'
                    )
                    SELECT      
                    au.user_id,
                    ap.product_id,
                    ROW_NUMBER() OVER (ORDER BY au.user_id, ap.product_id) AS row_id
                    FROM active_users au
                    CROSS JOIN active_products ap
                    """.format(db = db,
                               mmc_user_attributes = table_names["mmc_user_attributes"],
                               mmc_product_attributes = table_names["mmc_product_attributes"],
                               run_date = run_date,
                               user_lookback_win = user_lookback_win,
                               product_lookback_win = product_lookback_win)
    )

    shard_count = max(1, int(num_shards))
    if shard_count > 1:
        sdf = sdf.withColumn("shard_id", F.pmod(F.xxhash64("user_id"), F.lit(shard_count)))
    else:
        sdf = sdf.withColumn("shard_id", F.lit(0))
    
    sdf = sdf.withColumn("partition_date", F.lit(run_date))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(db+'.'+table_names["mmc_candidates"])

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

infer_config_path = dbutils.widgets.get("infer_config_path")
with open(infer_config_path, "r") as file:
    infer_config = json.load(file)

user_lookback_win = infer_config["user_lookback_win"]
product_lookback_win = infer_config["product_lookback_win"]

# dbutils.widgets.text("num_shards", "1")
num_shards = int(dbutils.widgets.get("num_shards") or "1")

In [0]:
create_user_product_candidate_set(spark, run_date, db, table_names, 
                                    user_lookback_win, product_lookback_win,
                                    num_shards)